# Plot

In [20]:
import pandas as pd
import matplotlib as mpl
mpl.use('Qt5Agg')
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import CheckButtons
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D
import matplotlib.transforms as transforms
from database.db_control import Database
import ipynbname
import os


### data import

In [21]:
db = Database(exp_name='DFFC_PdPtCu')

In [22]:

trial_index_list = [0]

df = None
for trial_index in trial_index_list:
    trial_df = pd.read_sql(
        f'SELECT * FROM active_learning.full_table WHERE arm_name LIKE \'{trial_index}\_%%\' '
        'AND abandoned IS NULL',
        db.dbConnection
    )
    df = trial_df if df is None else df.append(trial_df, ignore_index=True)

In [23]:
df

,trial_index,arm_name,sample_id,test_id,Pd,Pt,Cu,max_power,abandoned,abandon_reason,outlier
0,0,0_0,1,1,1.0,0.0,0.0,13.73000,None,None,None
1,0,0_1,2,2,2.0,1.0,0.0,32.13000,None,None,None
2,0,0_2,3,3,1.0,2.0,0.0,17.77000,None,None,None
3,0,0_3,4,4,0.0,1.0,0.0,0.76310,None,None,None
4,0,0_4,5,5,0.0,2.0,1.0,0.08309,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
128,0,0_17,56,123,5.0,1.0,5.0,11.30000,None,None,None
129,0,0_9,48,115,1.0,1.0,1.0,9.93200,None,None,None
130,0,0_2,41,127,1.0,2.0,0.0,5.95600,None,None,None
131,0,0_7,46,151,1.0,0.0,2.0,10.59000,None,None,None


In [24]:
df_sorted = df.sort_values(['test_id'])

In [25]:
df_sorted

,trial_index,arm_name,sample_id,test_id,Pd,Pt,Cu,max_power,abandoned,abandon_reason,outlier
0,0,0_0,1,1,1.0,0.0,0.0,13.73000,None,None,None
1,0,0_1,2,2,2.0,1.0,0.0,32.13000,None,None,None
2,0,0_2,3,3,1.0,2.0,0.0,17.77000,None,None,None
3,0,0_3,4,4,0.0,1.0,0.0,0.76310,None,None,None
4,0,0_4,5,5,0.0,2.0,1.0,0.08309,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
77,0,0_14,53,158,1.0,10.0,1.0,4.67400,None,None,None
87,0,0_15,54,159,1.0,1.0,10.0,6.02700,None,None,None
101,0,0_16,55,160,5.0,5.0,1.0,17.96000,None,None,None
91,0,0_17,56,161,5.0,1.0,5.0,25.15000,None,None,None


In [26]:
batch_size = 19
sample_repeats = 1
batch_sample = batch_size * sample_repeats

test_id_starting = df_sorted['test_id'].min()
test_id_starting_2 = 106

In [27]:
test_names = [
    '1st_test',
    '2nd_test',
    '3rd_test',
    '4th_test',
]
sample_names = [
    '1st_sample',
    '2nd_sample',
    # '3rd_sample',
]


exp_name = ipynbname.name()

save_dir = f'plots/{exp_name}'
os.makedirs(save_dir, exist_ok=True)

v_max = df['max_power'].max()
v_min = df['max_power'].min()

markers = {0: 'o', 1: '^', 2: 's', 3: '*', 4: 'D', 5: 'X'}
colors = [f'C{i}' for i in range(10)]

df_by_test = {}
for i in range(len(test_names)):
    # df_by_test[i] = df_sorted.iloc[i * batch_sample : (i + 1) * batch_sample, :]
    df_by_test[i] = pd.concat([df_sorted.loc[df_sorted['test_id'].between(test_id_starting + i * batch_sample, test_id_starting + (i + 1) * batch_sample - 1)], df_sorted.loc[df_sorted['test_id'].between(test_id_starting_2 + i * batch_sample, test_id_starting_2 + (i + 1) * batch_sample - 1)]])
    # df_by_test[i] = df_sorted.loc[df_sorted['test_id'].between(test_id_starting + i * batch_sample, test_id_starting + (i + 1) * batch_sample - 1)]

In [28]:
arm_dict = {trial_index: [f'{trial_index}_{arm}' for arm in range(batch_size)] for trial_index in trial_index_list}

df_by_test_arm = df_by_test.copy()

for i, test_df in df_by_test.items():
    for arm_list in arm_dict.values():
        arm_slice = {arm: pd.DataFrame() for arm in arm_list}
        for arm in arm_slice.keys():
            arm_slice[arm] = test_df[:][test_df.arm_name == arm].reset_index()
    df_by_test_arm[i] = arm_slice

In [29]:
# plot
plt.figure(figsize=(20, 8))

ax = plt.subplot(111)

offset = lambda x: transforms.ScaledTranslation(x / 72, 0, plt.gcf().dpi_scale_trans)
trans = ax.transData



for test_id, test_df in df_by_test_arm.items():

    scatter_plots = []
    line_plots = []
    for arm_name, test_arm_df in test_df.items():
        for i, test in test_arm_df.iterrows():
            scatter_plot = ax.scatter(test['arm_name'], test['max_power'], alpha=0.8,
                                      marker=markers[i],
                                      c=colors[test_id],
                                      s=45,
                                      # transform=trans+offset(6 * test_id)
                                      )
            scatter_plots.append(scatter_plot)


# common settings
ax.set_axisbelow(True)
ax.tick_params(axis='both', labelsize=13)

# axis x settings
ax.set_xlabel('arm_name')
ax.xaxis.label.set_size(15)

# axis y settings
ax.yaxis.grid(True, color='#EEEEEE')
ax.yaxis.label.set_size(15)

# legend setting
# ax.legend(scatter_plots, test_names, loc=0, prop={'size': 14})
# legend settings
legend_1 = [
    Line2D([0], [0], marker='o', color=colors[i], linestyle='None', label=test_names[i]) for i in range(len(test_names))
]
legend_2 = [
    Line2D([0], [0], marker=markers[i], color='k', linestyle='None', label=sample_names[i], alpha=0.4) for i in range(len(sample_names))
]

lgd_1 = plt.legend(handles=legend_1, loc=1)
plt.legend(handles=legend_2, loc=2, prop={'size': 10})
plt.gca().add_artist(lgd_1)

# subplot 2 axis y settings
ax.set_ylabel('max_power (mW)')
ax.set_ylim(v_min - 0.005, max(v_max * 1.1, 0))

# title
plot_title = f'{exp_name}'
ax.set_title(plot_title, fontsize=20, pad=20)

# saving
plt.savefig(f'{save_dir}/{plot_title}.jpg', dpi=300, bbox_inches='tight')

plt.close()

### plot settings

In [8]:


group_names = [
    '0A',
    '5A',
    '15A',
    '30A',
]

recipe_names = [
    'water 1h', 'KOH 1h', 'no dip'
]

groups = {}
group_interval = 3
group_count = 1
test_count = 3

recipe_count = len(recipe_names)
recipe_repeats = 1
recipes_sample_id = {}
recipes = {}
test_order_good = False



sample_id_starting = df['sample_id'].min()
starting_id = df['id'].min()

i_max = df['max_i'].max()
i_min = df['max_i'].min()


# alpha and color setting
test_round_alpha = {0: 0.8, 1: 0.4, 2: 0.1}
test_round_marker = {0: 'o', 1: '^', 2: 's', 3: '*', 4: 'D', 5: 'X'}
colors = [f'C{i}' for i in range(10)]

for i in range(recipe_count):
    recipes_sample_id[i] = [sample_id_starting + i + j * recipe_count for j in range(recipe_repeats)]
    recipes[i] = df.loc[df['sample_id'].apply(lambda x: any(k == x for k in recipes_sample_id[i]))]

KeyError: 'id'

## show recipe result one by one

In [ ]:
for recipe_id, recipe in recipes.items():

    tests = {}

    for i in range(test_count):
        if test_order_good:
            # if missing test but order is good
            tests[i] = recipe.loc[recipe['id'].between(starting_id + i * group_count * group_interval, starting_id + (i + 1) * group_count * group_interval - 1)]
        else:
            # if no missing test but test not in order
            recipe = recipe.sort_values(by=['sample_id', 'test_start_time'], ascending=[True, True])
            iloc = [j * test_count + i for j in range(recipe_repeats)]
            tests[i] = recipe.iloc[iloc, :]

    plt.figure(figsize=(10, 10))

    ax1 = plt.subplot(211)
    ax2 = plt.subplot(212)
    scatter_plots = []
    line_plots = []

    # plot max_i one by one
    for i, test in tests.items():
        scatter_plot = ax1.scatter(test['sample_id'], test['max_i'], alpha=0.8, c=colors[i])
        scatter_plots.append(scatter_plot)
        line_plot,  = ax1.plot(test['sample_id'], test['max_i'], linestyle='dashed', alpha=0.4, c=colors[i])
        line_plots.append(line_plot)

    # plot overpotential one by one
    for i, test in tests.items():
        ax2.scatter(test['sample_id'], test['overpotential'], alpha=0.8, c=colors[i])
        ax2.plot(test['sample_id'], test['overpotential'], linestyle='dashed', alpha=0.4, c=colors[i])

    for ax in [ax1, ax2]:
        ax.set_xlabel('sample index')
        ax.legend(line_plots, test_names, loc=0, prop={'size': 14})
        ax.xaxis.grid(True, color='#EEEEEE')
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f'))
        ax.set_axisbelow(True)
        ax.set_xticks(np.arange(recipe['sample_id'].min(), recipe['sample_id'].max() + 1, group_interval), Fontsize=15)
        ax.xaxis.label.set_size(15)
        ax.yaxis.label.set_size(15)
        ax.tick_params(axis='both', labelsize=13)

    plot_title = f'{exp_name}_{recipe_names[recipe_id]}'
    ax1.set_ylabel('max current (mA)')
    ax1.set_title(plot_title, fontsize=20, pad=20)
    ax1.set_ylim(min(i_min - 2, i_max - 40), i_max + 2)
    ax2.set_ylabel('overpotential (V)')
    ax2.set_ylim(v_min - 0.005, max(v_max + 0.005, v_min + 0.02))
    plt.savefig(f'{save_dir}/{plot_title}.png', dpi=300, bbox_inches='tight')
    plt.close()


## show recipe average by test

In [5]:
plt.figure(figsize=(10, 10))

ax1 = plt.subplot(211)
ax2 = plt.subplot(212)

for recipe_id, recipe in recipes.items():

    tests = {}

    for i in range(test_count):
        if test_order_good:
            # if missing test but order is good
            tests[i] = recipe.loc[recipe['id'].between(starting_id + i * group_count * group_interval, starting_id + (i + 1) * group_count * group_interval - 1)]
        else:
            # if no missing test but test not in order
            recipe = recipe.sort_values(by=['sample_id', 'test_start_time'], ascending=[True, True])
            iloc = [j * test_count + i for j in range(recipe_repeats)]
            tests[i] = recipe.iloc[iloc, :]

    # plot max_i one by one
    for i, test in tests.items():
        scatter_plot = ax1.scatter(recipe_names[recipe_id], test['max_i'].mean(), alpha=0.8, c=colors[i])
        ax1.errorbar(recipe_names[recipe_id], test['max_i'].mean(), yerr=test['max_i'].std(), c=colors[i], alpha=0.5, capsize=2)

    # plot overpotential one by one
    for i, test in tests.items():
        ax2.scatter(recipe_names[recipe_id], test['overpotential'].mean(), alpha=0.8, c=colors[i])
        ax2.errorbar(recipe_names[recipe_id], test['overpotential'].mean(), yerr=test['overpotential'].std(), c=colors[i], alpha=0.5, capsize=2)

for ax in [ax1, ax2]:
    ax.legend(scatter_plots, test_names, loc=0, prop={'size': 14})
    ax.xaxis.grid(True, color='#EEEEEE')
    ax.set_axisbelow(True)
    ax.yaxis.label.set_size(15)
    ax.tick_params(axis='both', labelsize=13)

plot_title = f'{exp_name}_recipe average by test'
ax1.set_ylabel('max current (mA)')
ax1.set_title(plot_title, fontsize=20, pad=20)
ax1.set_ylim(min(i_min - 2, i_max - 40), i_max + 2)
ax2.set_ylabel('overpotential (V)')
ax2.set_ylim(v_min - 0.005, max(v_max + 0.005, v_min + 0.02))
plt.savefig(f'{save_dir}/{plot_title}.png', dpi=300, bbox_inches='tight')
plt.close()

## show recipe result by test & group

In [7]:
for recipe_id, recipe in recipes.items():

    tests = {}

    for i in range(test_count):
        if test_order_good:
            # if missing test but order is good
            tests[i] = recipe.loc[recipe['id'].between(starting_id + i * group_count * group_interval, starting_id + (i + 1) * group_count * group_interval - 1)]
        else:
            # if no missing test but test not in order
            recipe = recipe.sort_values(by=['sample_id', 'test_start_time'], ascending=[True, True])
            iloc = [j * test_count + i for j in range(recipe_repeats)]
            tests[i] = recipe.iloc[iloc, :]

    plt.figure(figsize=(10, 10))

    ax1 = plt.subplot(211)
    ax2 = plt.subplot(212)
    scatter_plots = []
    line_plots = []

    # plot max_i one by one
    for i, test in tests.items():
        scatter_plot = ax1.scatter(group_names, test['max_i'], alpha=0.8, c=colors[i])
        scatter_plots.append(scatter_plot)
        line_plot,  = ax1.plot(group_names, test['max_i'], linestyle='dashed', alpha=0.4, c=colors[i])
        line_plots.append(line_plot)

    # plot overpotential one by one
    for i, test in tests.items():
        ax2.scatter(group_names, test['overpotential'], alpha=0.8, c=colors[i])
        ax2.plot(group_names, test['overpotential'], linestyle='dashed', alpha=0.4, c=colors[i])

    for ax in [ax1, ax2]:
        ax.set_xlabel('Groups')
        ax.legend(line_plots, test_names, loc=0, prop={'size': 14})
        ax.xaxis.grid(True, color='#EEEEEE')
        ax.set_axisbelow(True)
        ax.xaxis.label.set_size(15)
        ax.yaxis.label.set_size(15)
        ax.tick_params(axis='both', labelsize=13)

    plot_title = f'{exp_name}_{recipe_names[recipe_id]}'
    ax1.set_ylabel('max current (mA)')
    ax1.set_title(plot_title, fontsize=20, pad=20)
    ax1.set_ylim(min(i_min - 2, i_max - 40), i_max + 2)
    ax2.set_ylabel('overpotential (V)')
    ax2.set_ylim(v_min - 0.005, max(v_max + 0.005, v_min + 0.02))
    plt.savefig(f'{save_dir}/{plot_title}.png', dpi=300, bbox_inches='tight')
    plt.close()
